# Task 2.2

### Mapa del entorno
```
 S  F  F  F      →   estados  0  1  2  3
 F  H  F  H      →   estados  4  5  6  7
 F  F  F  H      →   estados  8  9 10 11
 H  F  F  G      →   estados 12 13 14 15
```
- S = Start
- F = Frozen (seguro)
- H = Hole (terminal, reward 0)
- G = Goal (terminal, reward +1)

### Dinámica estocástica
El hielo es resbaladizo: al elegir una acción, el agente se mueve en la dirección deseada con P = 1/3,
y con P = 1/3 hacia cada uno de los dos lados perpendiculares.

Esto replica lo definido en clase como función de transición T(s, a, s').

In [9]:
import numpy as np

## Task 2.1: Modelado del MDP Frozen Lake

Implementa los componentes del MDP vistos en clase:
- Estados : enteros 0–15
- Acciones : 0=Norte, 1=Sur, 2=Este, 3=Oeste
- T(s, a, s') : probabilidad de transición con estocasticidad del hielo
- R(s, a, s') : recompensa de la transición

In [10]:
class FrozenLakeMDP:
    # Definición del entorno

    GRID = [ # Mapa del lago: S=start, F=frozen, H=hole, G=goal
        'S', 'F', 'F', 'F',   # fila 0 → estados  0- 3
        'F', 'H', 'F', 'H',   # fila 1 → estados  4- 7
        'F', 'F', 'F', 'H',   # fila 2 → estados  8-11
        'H', 'F', 'F', 'G',   # fila 3 → estados 12-15
    ]

    N_STATES  = 16   # número total de estados
    N_ACTIONS = 4    # acciones: Norte, Sur, Este, Oeste
    GRID_SIZE = 4    # dimensión del grid

    ACTION_NAMES  = {0: 'Norte', 1: 'Sur', 2: 'Este', 3: 'Oeste'}
    ACTION_ARROWS = {0: '↑', 1: '↓', 2: '→', 3: '←'}

    # Desplazamiento (delta_fila, delta_col) por acción
    DELTAS = {
        0: (-1,  0),   # Norte: sube una fila
        1: ( 1,  0),   # Sur  : baja una fila
        2: ( 0,  1),   # Este : avanza una columna
        3: ( 0, -1),   # Oeste: retrocede una columna
    }

    # Acciones perpendiculares a cada dirección
    PERP = {
        0: [2, 3],   # Norte  → perpendiculares: Este, Oeste
        1: [2, 3],   # Sur    → perpendiculares: Este, Oeste
        2: [0, 1],   # Este   → perpendiculares: Norte, Sur
        3: [0, 1],   # Oeste  → perpendiculares: Norte, Sur
    }

    def __init__(self):
        # Estados terminales: hoyos (H) y la meta (G)
        # En un MDP: IsEnd(s) = True para estos estados
        self.terminal_states = {
            s for s, tile in enumerate(self.GRID)
            if tile in ('H', 'G')
        }
        # Matrices de transición y recompensa
        # T[s, a, s'] = P(s' | s, a)
        # R[s, a, s'] = Reward(s, a, s')
        self.T = np.zeros((self.N_STATES, self.N_ACTIONS, self.N_STATES))
        self.R = np.zeros((self.N_STATES, self.N_ACTIONS, self.N_STATES))

        self._build_transition_and_reward()

    # Conversión Estado ↔ Coordenadas

    def state_to_rc(self, s): # Estado lineal s → (fila, columna).
        return divmod(s, self.GRID_SIZE)

    def rc_to_state(self, r, c): # (fila, columna) → estado lineal.
        return r * self.GRID_SIZE + c

    def is_valid(self, r, c): # ¿Está (r,c) dentro del grid?
        return 0 <= r < self.GRID_SIZE and 0 <= c < self.GRID_SIZE

    # Función de movimiento

    def _move(self, s, a): # Intenta mover el agente desde s aplicando la acción a. Si el movimiento sale del grid, el agente se queda en s (choca contra la pared). Retorna el estado resultante s'.
        r, c = self.state_to_rc(s)
        dr, dc = self.DELTAS[a]
        nr, nc = r + dr, c + dc

        if self.is_valid(nr, nc):
            return self.rc_to_state(nr, nc)
        else:
            return s   # rebotar contra la pared

    # Construcción de T(s,a,s') Y R(s,a,s')

    def _build_transition_and_reward(self):
        for s in range(self.N_STATES):

            # Si el estado es terminal, no genera transiciones futuras
            if s in self.terminal_states:
                continue

            for a in range(self.N_ACTIONS):

                # Las tres direcciones posibles con P=1/3 cada una
                possible = [a] + self.PERP[a]  # [deseada, perp1, perp2]

                for act in possible:
                    s_prime = self._move(s, act)

                    # Acumulamos probabilidad
                    # (distintas acciones pueden llevar al mismo s')
                    self.T[s, a, s_prime] += 1/3

                    # Reward(s, a, s') = +1 solo si s' es el Goal
                    if self.GRID[s_prime] == 'G':
                        self.R[s, a, s_prime] = 1.0

        # Sanity check: sum_s' T(s,a,s') = 1 para estados no terminales
        for s in range(self.N_STATES):
            if s not in self.terminal_states:
                total = self.T[s].sum(axis=1)  # suma sobre s' para cada a
                assert np.allclose(total, 1.0), \
                    f"T no suma 1 en estado {s}: {total}"

    # Interfaz Publica usada por el algoritmo de Value Iteration.

    def get_transitions(self, s, a): # Retorna lista de (probabilidad, s', recompensa) para T(s,a,·).
        return [
            (self.T[s, a, sp], sp, self.R[s, a, sp])
            for sp in range(self.N_STATES)
            if self.T[s, a, sp] > 0
        ]

    def is_terminal(self, s):
        return s in self.terminal_states

    def render_grid(self): # Imprime el mapa del lago.
        print("Mapa del lago")
        for r in range(self.GRID_SIZE):
            print(' '.join(f' {self.GRID[self.rc_to_state(r,c)]} '
                           for c in range(self.GRID_SIZE)))
        print()

    def print_transition_example(self, s, a): # Muestra T(s,a,·) para verificación manual.
        r, c = self.state_to_rc(s)
        print(f"Transiciones: estado {s} ({r},{c})='{self.GRID[s]}',"
              f" acción '{self.ACTION_NAMES[a]}'")
        print(f"  {'s\'':>4}  {'tile':>4}  {'prob':>6}  {'reward':>6}")
        for prob, sp, rew in self.get_transitions(s, a):
            rp, cp = self.state_to_rc(sp)
            print(f"  {sp:>4} ({rp},{cp}) {self.GRID[sp]:>4}  "
                  f"{prob:>6.4f}  {rew:>6.2f}")
        print()

### Instanciación

In [11]:
mdp = FrozenLakeMDP()
mdp.render_grid()

print(f"Estados terminales (H y G): {sorted(mdp.terminal_states)}")
print(f"Forma de T : {mdp.T.shape}  → (estados x acciones x estados)")
print(f"Forma de R : {mdp.R.shape}  → (estados x acciones x estados)")

Mapa del lago
 S   F   F   F 
 F   H   F   H 
 F   F   F   H 
 H   F   F   G 

Estados terminales (H y G): [5, 7, 11, 12, 15]
Forma de T : (16, 4, 16)  → (estados x acciones x estados)
Forma de R : (16, 4, 16)  → (estados x acciones x estados)


### Pruebas para ver si funciona (borrar despues)

In [12]:
# --- Caso 1: Start (0,0) yendo Norte ---
# Norte choca con la pared superior → se queda en 0
# Las perpendiculares (Este/Oeste) llevan a 1 o también rebotan a 0
mdp.print_transition_example(s=0, a=0)

# --- Caso 2: estado 1 (0,1) yendo Sur ---
# Sur → estado 5 (H), Este → estado 2, Oeste → estado 0
mdp.print_transition_example(s=1, a=1)

# --- Caso 3: estado 14 (3,2) yendo Este → puede llegar al Goal ---
# Este → estado 15 (G, reward +1), Norte → 10, Sur rebota a 14
mdp.print_transition_example(s=14, a=2)

Transiciones: estado 0 (0,0)='S', acción 'Norte'
    s'  tile    prob  reward
     0 (0,0)    S  0.6667    0.00
     1 (0,1)    F  0.3333    0.00

Transiciones: estado 1 (0,1)='F', acción 'Sur'
    s'  tile    prob  reward
     0 (0,0)    S  0.3333    0.00
     2 (0,2)    F  0.3333    0.00
     5 (1,1)    H  0.3333    0.00

Transiciones: estado 14 (3,2)='F', acción 'Este'
    s'  tile    prob  reward
    10 (2,2)    F  0.3333    0.00
    14 (3,2)    F  0.3333    0.00
    15 (3,3)    G  0.3333    1.00

